In [1]:
%matplotlib inline
import pandas as pd
import glob
import matplotlib.pyplot as plt
import os
import numpy as np
import random
import scipy.stats as st
from collections import defaultdict
import seaborn as sns
import csv
import math
from scipy.stats import ttest_1samp
from scipy.stats import chi2
from scipy.spatial.distance import mahalanobis
import copy
from scipy.interpolate import PchipInterpolator
from scipy.signal import savgol_filter
from mpl_toolkits import mplot3d
import matplotlib.cm as cm
from matplotlib.collections import LineCollection
from matplotlib.colors import LogNorm
from scipy.stats import zscore

In [2]:
def remove_outliers_prob1(signal, max_iters=10, prob_thresh=1e-3, neighbor_window=3):
    """
    Removes outliers using Mahalanobis on (x, diff(x)) space.
    signal = [(value, original_index)]
    Returns full-length list where removed indices hold NaN.
    """
    values = np.array([v for v, idx in signal], float)
    orig_idx = np.array([idx for v, idx in signal], int)

    if len(values) < 3:
        return [(values[i], orig_idx[i]) for i in range(len(values))]

    diffs = np.diff(values)
    p_space = np.column_stack((values[:-1], diffs))
    p_orig_idx = orig_idx[:-1].copy()

    for _ in range(max_iters):

        if len(p_space) < 3:
            break

        cov = np.cov(p_space.T)
        if np.linalg.det(cov) == 0:
            break

        inv = np.linalg.inv(cov)
        mean = np.mean(p_space, axis=0)
        m_dist = np.array([mahalanobis(pt, mean, inv) for pt in p_space])
        p_vals = chi2.sf(m_dist**2, df=2)

        remove_mask = p_vals < prob_thresh
        if not np.any(remove_mask):
            break

        remove_orig_indices = p_orig_idx[remove_mask]

        expand = set(remove_orig_indices)
        for idx in remove_orig_indices:
            for off in range(-neighbor_window, neighbor_window + 1):
                expand.add(idx + off)

        expand = {i for i in expand if i in orig_idx[:-1]}

        keep_mask = ~np.isin(p_orig_idx, list(expand))
        p_space = p_space[keep_mask]
        p_orig_idx = p_orig_idx[keep_mask]

    cleaned = np.full_like(values, np.nan, dtype=float)

    cleaned[np.isin(orig_idx, p_orig_idx)] = values[np.isin(orig_idx, p_orig_idx)]

    if orig_idx[-1] not in p_orig_idx:
        cleaned[-1] = values[-1]
    return [(cleaned[i], orig_idx[i]) for i in range(len(values))]

In [3]:
def linear_interpolation2(signal):
    """
    Linearly interpolates NaNs in a signal where each entry is (value, original_index).
    Returns list of (interpolated_value, original_index).

    NaNs at the beginning or end remain NaN.
    """
    signal = np.array(signal, dtype=object)
    values = np.array([v for v, idx in signal], dtype=float)
    orig_idx = np.array([idx for v, idx in signal], dtype=int)

    valid_mask = ~np.isnan(values)
    if np.sum(valid_mask) < 2:
        # Not enough to interpolate → return original
        return [(values[i], orig_idx[i]) for i in range(len(values))]

    x = np.arange(len(values))
    valid_x = x[valid_mask]
    valid_y = values[valid_mask]

    interpolated = np.interp(x, valid_x, valid_y, left=np.nan, right=np.nan)

    interpolated[valid_mask] = values[valid_mask]

    return [(interpolated[i], orig_idx[i]) for i in range(len(values))]

In [4]:
def pchip_interpolate2(signal, time=None):
    """
    PCHIP interpolate a signal where each element is (value, original_index),
    preserving original indices in output.

    Returns:
        list of (interpolated_value, original_index)
    """

    signal = np.array(signal, dtype=object)
    values = np.array([v for v, idx in signal], dtype=float)
    orig_idx = np.array([idx for v, idx in signal], dtype=int)

    if time is None:
        time = np.arange(len(values))
    else:
        time = np.array(time)
        
    valid_mask = ~np.isnan(values)

    if np.sum(valid_mask) < 2:
        # Not enough data → return original (with NaNs intact)
        return [(values[i], orig_idx[i]) for i in range(len(values))]

    valid_times = time[valid_mask]
    valid_data  = values[valid_mask]

    pchip = PchipInterpolator(valid_times, valid_data)
    interp = pchip(time)
    interp[valid_mask] = values[valid_mask]

    return [(interp[i], orig_idx[i]) for i in range(len(values))]

In [30]:
def total_outlier_removal(signal):
    '''
    Applies two levels of probability outlier removal
    One with a pchip interpolation and one with linear interpolation
    
    I commented out the interpolation code because sometimes it messes it up. 
    I would recommend first just getting the data without the outliers removed, 
    note down the frame values where it starts and ends, and then play around with the interpolation code.
    '''
    mahal1 = remove_outliers_prob1(signal, neighbor_window=2)
    cleaned1 = pchip_interpolate2(mahal1)

    if np.all(np.isnan(mahal1)):
        return np.array([])
        
    mahal2 = remove_outliers_prob1(cleaned1, neighbor_window=2)
    cleaned2 = linear_interpolation2(mahal2)
    
    return cleaned2

In [21]:
def clean_gaze_coords(coords_dict, 
                      x_min=0, x_max=1800, 
                      y_min=200, y_max=1500,
                      z_thresh=7):
    """
    Removes gaze outliers by applying:
    1. Screen boundary filtering (x_min/x_max, y_min/y_max)
    2. Z-score filtering per dimension (x or y too extreme)
    """

    if len(coords_dict) == 0:
        return {}

    frames = np.array(list(coords_dict.keys()))
    coords = np.atleast_2d(np.array(list(coords_dict.values()), dtype=float))
    
    # --- 1. Remove points outside screen bounds ---
    in_bounds_mask = (
        (coords[:,0] >= x_min) & (coords[:,0] <= x_max) &
        (coords[:,1] >= y_min) & (coords[:,1] <= y_max)
    )
    frames = frames[in_bounds_mask]
    coords = coords[in_bounds_mask]
    
    # --- 2. Z-score filtering ---
    if len(coords) >= 5:
        z_x = np.abs(zscore(coords[:,0]))
        z_y = np.abs(zscore(coords[:,1]))
        good_mask = (z_x < z_thresh) & (z_y < z_thresh)
        frames = frames[good_mask]
        coords = coords[good_mask]
    
    # --- 3. Reconstruct dictionary ---
    cleaned_dict = {frame: tuple(coord) for frame, coord in zip(frames, coords)}
    
    return cleaned_dict

In [11]:
results = sorted(os.listdir('Contrast_Pupil_Static')) ###input the name of the folder with all of your deep lab cut files here
data = {}

for file in results:
    data[file] = {}
    if file[0] == 'M':
        for x in sorted(os.listdir('Contrast_Pupil_Static/' + file)):
            if x[5] =='D':
                run = x[0:5]
            else:
                run = x[0:6]
            df = pd.read_csv('Contrast_Pupil_Static/' + file + '/' + x, header=None)
            df_dropped = df.drop([0, 1, 2])
            df_reset = df_dropped.reset_index(drop=True)
            df_reset.drop(columns=df.columns[0], 
            axis=1, 
            inplace=True)
            df_reset.columns = ['top_x', 'top_y', 'top_likelihood', 'bottom_x', 'bottom_y', 'bottom_likelihood', 
                                'anterior_x', 'anterior_y', 'anterior_likelihood',
                                'posterior_x', 'posterior_y', 'posterior_likelihood']
            df_float = df_reset.astype(float)
            df_reset['frame_index']= df_reset.index
            
            data[file][run] = df_float

del data['.DS_Store']
del data['.ipynb_checkpoints']

In [13]:
#creates cleaned data that removes all rows that have a likelihood under 0.9
#this only works for the x, y corrdinate data from deeplab cut that includes the likelihood values
#if you already have a list of your
cleaned_data = copy.deepcopy(data)
for df_dict in cleaned_data:
    for df in cleaned_data[df_dict]:
        for index, row in cleaned_data[df_dict][df].iterrows():
            likelihood_avg = (float(row['top_likelihood']) + float(row['bottom_likelihood']) + float(row['anterior_likelihood']) + float(row['posterior_likelihood']))/4
            if likelihood_avg < 0.9:
                cleaned_data[df_dict][df].loc[index, :] = np.nan
            if float(row['posterior_x']) - float(row['anterior_x']) < 0:
                cleaned_data[df_dict][df].loc[index, :] = np.nan
        cleaned_data[df_dict][df]['frame_index']= cleaned_data[df_dict][df].index

In [31]:
#this just makes the dictionaries for your pupil size from the deeplab cut raw data files
#don't need to run this if you already have your data
pupil_dilation_dict = {}
x_and_y_pupil_dict = {}
for days in cleaned_data:
    pupil_dilation_dict[days] = {}
    x_and_y_pupil_dict[days] = {}
    for sessions in cleaned_data[days]:
        pupil_dilation_dict[days][sessions] = {}
        x_and_y_pupil_dict[days][sessions] = {}
        for index, row in cleaned_data[days][sessions].iterrows():
            pupil_dilation_dict[days][sessions][index] = (math.sqrt((float(row['posterior_x']) - float(row['anterior_x']))**2 + (float(row['posterior_y']) - float(row['anterior_y']))**2), row['frame_index'])
            x_and_y_pupil_dict[days][sessions][index] = (((float(row['posterior_x']) + float(row['anterior_x']))/2, (float(row['anterior_y']) + float(row['posterior_y']))/2), row['frame_index'])

In [32]:
#runs the outlier removal on the dictionary of pupil sizes
pupil_mahal_dict = {}
for days in pupil_dilation_dict:
    pupil_mahal_dict[days] = {}
    for sessions in pupil_dilation_dict[days]:
        signal = list(pupil_dilation_dict[days][sessions].values())
        pupil_mahal_dict[days][sessions] = total_outlier_removal(signal)

for days in pupil_mahal_dict:
    for sessions in pupil_mahal_dict[days]:
        cleaned = []
        for v in pupil_mahal_dict[days][sessions]:
            if not np.isnan(v[0]):
                cleaned.append(v)
        pupil_mahal_dict[days][sessions] = cleaned

In [ ]:
x_y_mahal_dict = {}
for days in x_and_y_pupil_dict:
    x_y_mahal_dict[days] = {}
    for sessions in x_and_y_pupil_dict[days]:
        x_y_mahal_dict[days][sessions] = {}
        signal1 = [(x[0][0],x[1]) for x in list(x_and_y_pupil_dict[days][sessions].values())]
        signal2 = [(x[0][1],x[1]) for x in list(x_and_y_pupil_dict[days][sessions].values())]
        #abs_thresh1 = find_abs_thresh(signal1)
        #abs_thresh2 = find_abs_thresh(signal2)
        x_y_mahal_dict[days][sessions]['x_coords'] = remove_outliers_prob1(signal1)
        x_y_mahal_dict[days][sessions]['y_coords'] = remove_outliers_prob1(signal2)

In [22]:
clipped_x_y_dict = {}
for days in x_y_mahal_dict:
    clipped_x_y_dict[days] = {}
    for sessions in x_y_mahal_dict[days]:
        clipped_x_y_dict[days][sessions] = {}
        for coordinate in x_y_mahal_dict[days][sessions]:
            old_list = x_y_mahal_dict[days][sessions][coordinate]
            new_list = [v for v in old_list if not np.isnan(v[0])]
            clipped_x_y_dict[days][sessions][coordinate] = new_list

final_x_y_dict = {}
for days in clipped_x_y_dict:
    final_x_y_dict[days] = {}
    for sessions in clipped_x_y_dict[days]:
        temp_dict = {}
        final_x_y_dict[days][sessions] = {}
        
        x_list = clipped_x_y_dict[days][sessions]["x_coords"]
        y_list = clipped_x_y_dict[days][sessions]["y_coords"]

        # Convert to dicts keyed by frame
        x_dict = {frame: x for (x, frame) in x_list}
        y_dict = {frame: y for (y, frame) in y_list}

        # Find frames that exist in BOTH x and y
        common_frames = set(x_dict.keys()) & set(y_dict.keys())

        # Build the new dictionary
        for frame in sorted(common_frames):
            temp_dict[frame] = (x_dict[frame], y_dict[frame])
        
        final_x_y_dict[days][sessions] = clean_gaze_coords(temp_dict)

In [33]:
#gives you the data in an excel file, may want to play around with this
with pd.ExcelWriter("contrast_pupil_size_data.xlsx", engine="xlsxwriter") as writer:
    for day, sessions in pupil_mahal_dict.items():
        day_df = pd.DataFrame()

        for session, pupil_arr in sessions.items():

            # Convert list of tuples → 2 arrays
            values = [v for v, idx in pupil_arr]
            indices = [idx for v, idx in pupil_arr]

            session_df = pd.DataFrame({
                f"{session}_frame": indices,
                f"{session}_pupil_size": values
            })

            day_df = pd.concat([day_df, session_df], axis=1)

        day_df.to_excel(writer, sheet_name=day, index=False)

In [24]:
with pd.ExcelWriter("contrast_x_y_ouliers_removed.xlsx", engine="xlsxwriter") as writer:
    for day, sessions in final_x_y_dict.items():
        # Create an empty DataFrame for this day
        day_df = pd.DataFrame()

        for session, coords in sessions.items():
            
            x = [x for (x, y) in list(coords.values())]
            y = [y for (x, y) in list(coords.values())]

            # Make frame numbers
            frames = list(coords.keys())

            # Build session dataframe
            session_df = pd.DataFrame({
                f"{session}_frame": frames,
                f"{session}_x": x,
                f"{session}_y": y
            })

            # Concatenate horizontally
            day_df = pd.concat([day_df, session_df], axis=1)

        # Save this day's sheet
        day_df.to_excel(writer, sheet_name=day, index=False)